### Description
Develop a basic content recommendation model based on
Netflix titles and categories.

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
df=pd.read_csv('../../data/cleaned/dataset_cleaned.csv')
df.shape

(8790, 12)

In [2]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,US,2021-09-25,2020,PG-13,90 min,Documentaries,90,minutes
1,s3,TV Show,Ganglands,Julien Leclercq,FR,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",1,seasons
2,s6,TV Show,Midnight Mass,Mike Flanagan,US,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",1,seasons
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,BR,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",91,minutes
4,s8,Movie,Sankofa,Haile Gerima,US,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",125,minutes


In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define the dataset based on your provided rows
data = {
    'show_id': ['s1', 's3', 's6', 's14', 's8'],
    'type': ['Movie', 'TV Show', 'TV Show', 'Movie', 'Movie'],
    'title': [
        'Dick Johnson Is Dead', 
        'Ganglands', 
        'Midnight Mass', 
        'Confessions of an Invisible Girl', 
        'Sankofa'
    ],
    'director': ['Kirsten Johnson', 'Julien Leclercq', 'Mike Flanagan', 'Bruno Garotti', 'Haile Gerima'],
    'country': ['US', 'FR', 'US', 'BR', 'US'],
    'release_year': [2020, 2021, 2021, 2021, 1993],
    'rating': ['PG-13', 'TV-MA', 'TV-MA', 'TV-PG', 'TV-MA'],
    'listed_in': [
        'Documentaries', 
        'Crime TV Shows, International TV Shows, TV Action', 
        'TV Dramas, TV Horror, TV Mysteries', 
        'Children & Family Movies, Comedies', 
        'Dramas, Independent Movies, International Movies'
    ]
}

# Create the DataFrame
df = pd.DataFrame(data)
df


,show_id,type,title,director,country,release_year,rating,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,US,2020,PG-13,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,FR,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Action"
2,s6,TV Show,Midnight Mass,Mike Flanagan,US,2021,TV-MA,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,BR,2021,TV-PG,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,US,1993,TV-MA,"Dramas, Independent Movies, International Movies"


In [8]:
# Create a metadata "soup" by combining content features
def create_soup(row):
    # Lowercase and remove spaces from directors to treat full names as a single token
    director = str(row['director']).lower().replace(" ", "")
    country = str(row['country']).lower().replace(" ", "")
    categories = str(row['listed_in']).lower()
    
    return f"{director} {country} {categories}"

# Apply the function to create the soup column
df['soup'] = df.apply(create_soup, axis=1)
df[['title', 'soup']]


,title,soup
0,Dick Johnson Is Dead,kirstenjohnson us documentaries
1,Ganglands,"julienleclercq fr crime tv shows, internationa..."
2,Midnight Mass,"mikeflanagan us tv dramas, tv horror, tv myste..."
3,Confessions of an Invisible Girl,"brunogarotti br children & family movies, come..."
4,Sankofa,"hailegerima us dramas, independent movies, int..."


In [9]:
# Initialize TF-IDF Vectorizer and remove English stop words
tfidf = TfidfVectorizer(stop_words='english')

# Construct the TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(df['soup'])

# Calculate the Cosine Similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a reverse mapping of titles to indices for quick lookups
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
print("Similarity matrix calculated successfully. Shape:", cosine_sim.shape)


Similarity matrix calculated successfully. Shape: (5, 5)


In [13]:
def get_recommendations(title, cosine_sim=cosine_sim, num_recommendations=2):
    # Check if the title exists in the dataset
    if title not in indices:
        return f"Title '{title}' not found in the dataset."
    
    # Get the index of the title
    idx = indices[title]
    
    # Get pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # CRITICAL FIX: Sort using the score (x[1]) instead of the whole tuple
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top matches (excluding the input title itself at index 0)
    sim_scores = sim_scores[1:num_recommendations+1]
    
    # CRITICAL FIX: Properly extract the index [0] and the score [1]
    movie_indices = [element[0] for element in sim_scores]
    scores = [element[1] for element in sim_scores]
    
    # Build a clean output DataFrame
    return pd.DataFrame({
        'Recommended Title': df['title'].iloc[movie_indices].values,
        'Categories': df['listed_in'].iloc[movie_indices].values,
        'Similarity Score': scores
    })


In [14]:
# Test case: Get recommendations for 'Midnight Mass'
target_title = 'Midnight Mass'
results = get_recommendations(target_title, num_recommendations=2)

print(f"Recommendations for '{target_title}':")
results


Recommendations for 'Midnight Mass':


,Recommended Title,Categories,Similarity Score
0,Ganglands,"Crime TV Shows, International TV Shows, TV Action",0.498741
1,Sankofa,"Dramas, Independent Movies, International Movies",0.086861
